In [1]:
import os, sys

nb_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(nb_dir, '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print('Using project root:', project_root)
print('First sys.path entry:', sys.path[0])

Using project root: /home/rithvik/Documents/hnrs/Decoder
First sys.path entry: /home/rithvik/Documents/hnrs/Decoder


In [2]:
import numpy as np
from scipy.sparse import csr_matrix, eye, hstack, save_npz, load_npz
import scipy.io

In [3]:
from utils.LDPC_encode import LDPCEncode
from utils.awgn_channel import AWGNChannel
from utils.find_ber import findBER

In [4]:
from ldpc.bp_decoder import BpDecoder

In [5]:
H_mat_dat = scipy.io.loadmat('H.mat')
H = csr_matrix(H_mat_dat['H'])

In [6]:
P = np.array([[16, 17, 22, 24,  9,  3, 14, -1,  4,  2,  7, -1, 26, -1,  2, -1, 21, -1,  1,  0, -1, -1, -1, -1],
     [25, 12, 12,  3,  3, 26,  6, 21, -1, 15, 22, -1, 15, -1,  4, -1, -1, 16, -1,  0,  0, -1, -1, -1],
     [25, 18, 26, 16, 22, 23,  9, -1,  0, -1,  4, -1,  4, -1,  8, 23, 11, -1, -1, -1,  0,  0, -1, -1],
      [ 9,  7,  0,  1, 17, -1, -1,  7,  3, -1,  3, 23, -1, 16, -1, -1, 21, -1,  0, -1, -1,  0,  0, -1],
     [24,  5, 26,  7,  1, -1, -1, 15, 24, 15, -1,  8, -1, 13, -1, 13, -1, 11, -1, -1, -1, -1,  0,  0],
      [ 2,  2, 19, 14, 24,  1, 15, 19, -1, 21, -1,  2, -1, 24, -1,  3, -1,  2,  1, -1, -1, -1, -1,  0]])

In [7]:
print(P.shape)

(6, 24)


In [8]:
decoder = BpDecoder(H, schedule="cluster")

In [9]:
n = 486
n_frames = 10000


message = np.zeros((n_frames, n), dtype=int)
print('Message Shape: ', message.shape)

Message Shape:  (10000, 486)


In [10]:
encoded_codeword = LDPCEncode(message)
print("Encoded codeword shape:", encoded_codeword.shape) 

tx_codeword = 1 - 2 * encoded_codeword 

Encoded codeword shape: (10000, 648)


In [11]:
m, _ = H.shape
arr = np.arange(m)
clusters = arr.reshape(6, -1)

In [12]:
snrs = [-3, -2, -1, 0, 1, 2, 3, 4, 5]
bers = []

for snr in snrs:
    ebno = snr * 4 / 3
    schedule = decoder.m2i2_scheduler(P, code_rate= 0.75, EbN0=ebno, max_iterations=30)

    rx_llrs = AWGNChannel(tx_codeword, snr_db=snr)
    decoded_codewords = []

    for i in range(n_frames):
        llr = rx_llrs[i, :]

        decoder.reset()
        decoder.initialise_log_domain_bp(llr)

        for idx in schedule:
            scheduled_cluster = clusters[idx]
            llr = decoder.decode_cluster(scheduled_cluster)
        
        decoded_codeword = (llr < 0).astype(int)
        decoded_codewords.append(decoded_codeword)
    
    decoded_codewords = np.array(decoded_codewords)
    decoded_message = decoded_codewords[:, :n]
    ber = findBER(message, decoded_message)
    bers.append(ber)
    print(f"BER at SNR {snr} dB: {ber}")
    print('Schedule Shape: ', schedule.shape)
    print('Schedule: ', schedule)

BER at SNR -3 dB: 0.15785637860082305
Schedule Shape:  (1,)
Schedule:  [0]
BER at SNR -2 dB: 0.13027777777777777
Schedule Shape:  (1,)
Schedule:  [0]
BER at SNR -1 dB: 0.10357510288065844
Schedule Shape:  (1,)
Schedule:  [0]
BER at SNR 0 dB: 0.07753271604938272
Schedule Shape:  (30,)
Schedule:  [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
BER at SNR 1 dB: 0.031493621399176955
Schedule Shape:  (30,)
Schedule:  [3 4 5 4 1 0 2 3 5 0 1 2 0 4 1 0 1 0 1 4 0 1 0 1 0 1 0 1 0 1]
BER at SNR 2 dB: 0.011328395061728395
Schedule Shape:  (30,)
Schedule:  [3 4 5 2 0 1 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3]
BER at SNR 3 dB: 2.5308641975308642e-05
Schedule Shape:  (30,)
Schedule:  [0 1 2 5 4 3 0 1 5 2 4 3 0 1 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5]
BER at SNR 4 dB: 0.0001308641975308642
Schedule Shape:  (30,)
Schedule:  [0 5 2 1 4 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3]
BER at SNR 5 dB: 0.0
Schedule Shape:  (30,)
Schedule:  [0 4 2 3 5 1 3 5 2 1 3 5 2 1 3 5 2 1 3 5 2 1 3 5